# Flang REPL stress test and OpenBLAS/LAPACK interop

This notebook exercises persistent state, procedures, recursion, modules, `ISO_C_BINDING`, native library loading, CBLAS, and LAPACKE in one live `flangInterpreter` session.

The OpenBLAS examples expect the library in the active environment. With conda-forge, install it using `micromamba install -c conda-forge openblas`. On macOS the library is `libopenblas.dylib`; on Linux, change the `%load` cell to `libopenblas.so`.

## Persistent state and redefinition

A declaration becomes session state. A later declaration shadows it, while procedures compiled earlier retain their original binding.

In [1]:
integer :: counter
counter = 3

In [2]:
subroutine print_old_counter()
  print *, counter
end subroutine

In [3]:
integer :: counter
counter = 100

In [4]:
call print_old_counter()
print *, counter

 3
 100


## Procedures, recursion, and complete modules

These are complete standard Fortran program units. They are compiled as definition-only cells and become visible to later cells.

In [5]:
recursive function factorial(n) result(value)
  integer, intent(in) :: n
  integer :: value
  if (n <= 1) then
    value = 1
  else
    value = n * factorial(n - 1)
  end if
end function

In [6]:
factorial(10)

 3628800


In [7]:
module vector_math
contains
  pure integer function cube(x)
    integer, intent(in) :: x
    cube = x*x*x
  end function
end module

In [8]:
use vector_math
print *, cube(7)

 343


## `ISO_C_BINDING` and a cross-cell C ABI symbol

In [9]:
use iso_c_binding
integer(c_int), target :: c_value
integer(c_int), pointer :: c_alias
type(c_ptr) :: c_address
c_value = 41
c_address = c_loc(c_value)
call c_f_pointer(c_address, c_alias)
c_alias = c_alias + 1
print *, c_value

 42


In [10]:
function c_square(x) result(value) bind(c, name="stress_c_square")
  use iso_c_binding
  integer(c_int), value :: x
  integer(c_int) :: value
  value = x * x
end function

In [11]:
print *, c_square(12)

 144


## Load OpenBLAS

`%load` publishes a native dynamic library to the same process and symbol namespace used by the JIT. A bare library name keeps the notebook independent of the environment prefix.

In [12]:
%load libopenblas.dylib

loaded 'libopenblas.dylib'


## CBLAS matrix multiplication

The interface uses the stable C ABI rather than compiler-specific Fortran module files. The expected product is `[[19, 22], [43, 50]]`.

In [13]:
subroutine run_cblas_dgemm()
  use iso_c_binding
  interface
    subroutine cblas_dgemm(layout, transa, transb, m, n, k, &
        alpha, a, lda, b, ldb, beta, c, ldc) &
        bind(c, name="cblas_dgemm")
      import c_int, c_double
      integer(c_int), value :: layout, transa, transb
      integer(c_int), value :: m, n, k, lda, ldb, ldc
      real(c_double), value :: alpha, beta
      real(c_double), intent(in) :: a(*), b(*)
      real(c_double), intent(inout) :: c(*)
    end subroutine
  end interface
  real(c_double) :: a(2,2), b(2,2), c(2,2)
  a = reshape([1.0_c_double, 3.0_c_double, &
               2.0_c_double, 4.0_c_double], [2,2])
  b = reshape([5.0_c_double, 7.0_c_double, &
               6.0_c_double, 8.0_c_double], [2,2])
  c = 0.0_c_double
  call cblas_dgemm(102_c_int, 111_c_int, 111_c_int, &
      2_c_int, 2_c_int, 2_c_int, 1.0_c_double, a, 2_c_int, &
      b, 2_c_int, 0.0_c_double, c, 2_c_int)
  print *, nint(c(1,1)), nint(c(1,2)), &
           nint(c(2,1)), nint(c(2,2))
end subroutine

In [14]:
call run_cblas_dgemm()

 19 22 43 50


## LAPACKE linear solve

This solves `3x + y = 9` and `x + 2y = 8`. The expected result is `info = 0`, `x = 2`, `y = 3`.

In [15]:
subroutine run_lapack_dgesv()
  use iso_c_binding
  interface
    function lapacke_dgesv(layout, n, nrhs, a, lda, ipiv, b, ldb) &
        result(info) bind(c, name="LAPACKE_dgesv")
      import c_int, c_double
      integer(c_int), value :: layout, n, nrhs, lda, ldb
      real(c_double), intent(inout) :: a(*), b(*)
      integer(c_int), intent(out) :: ipiv(*)
      integer(c_int) :: info
    end function
  end interface
  real(c_double) :: a(2,2), b(2)
  integer(c_int) :: ipiv(2), info
  a = reshape([3.0_c_double, 1.0_c_double, &
               1.0_c_double, 2.0_c_double], [2,2])
  b = [9.0_c_double, 8.0_c_double]
  info = lapacke_dgesv(102_c_int, 2_c_int, 1_c_int, a, &
                         2_c_int, ipiv, b, 2_c_int)
  print *, info, nint(b(1)), nint(b(2))
end subroutine

In [16]:
call run_lapack_dgesv()

 0 2 3


## Inspect without committing state

`%%mlir` prints the LLVM-dialect module prepared for the JIT, then discards the provisional cell. The next cell can continue using the previous committed state.

In [17]:
%%mlir
integer :: inspection_only
inspection_only = 42

module @flang_repl_module_17 attributes {dlti.dl_spec = #dlti.dl_spec<!llvm.ptr<270> = dense<32> : vector<4xi64>, !llvm.ptr<271> = dense<32> : vector<4xi64>, !llvm.ptr<272> = dense<64> : vector<4xi64>, i64 = dense<64> : vector<2xi64>, i128 = dense<128> : vector<2xi64>, !llvm.ptr = dense<64> : vector<4xi64>, i1 = dense<8> : vector<2xi64>, i8 = dense<8> : vector<2xi64>, i16 = dense<16> : vector<2xi64>, i32 = dense<32> : vector<2xi64>, f16 = dense<16> : vector<2xi64>, f64 = dense<64> : vector<2xi64>, f128 = dense<128> : vector<2xi64>, "dlti.endianness" = "little", "dlti.mangling_mode" = "o", "dlti.legal_int_widths" = array<i32: 32, 64>, "dlti.stack_alignment" = 128 : i64, "dlti.function_pointer_alignment" = #dlti.function_pointer_alignment<32, function_dependent = true>>, fir.allocation_policy = #fir.allocation_policy<stack_arrays = false, small_array_threshold = 1024, total_stack_limit = 4194304>, fir.defaultkind = "a1c4d8i4l4r4", fir.kindmap = "", fir.relocation_model = 1 : i32, llvm.da

In [18]:
print *, counter

 100
